# Recursive Confabulation — Reproduction Notebook

**Author:** Bentley DeVilling (Course Correct Labs)  
**Email:** Bentley@CourseCorrectLabs.com

This notebook reproduces the core findings from the Recursive Confabulation study. It:
- Installs pinned dependencies
- Loads CSV data from the repository
- Recomputes statistical tests
- Generates verification figures

Outputs are saved to `/content/figures/`


## 🚀 Quick Start

**To reproduce the full analysis:**
1. Click **Runtime → Run all** (or press Ctrl+F9 / Cmd+F9)
2. Wait for all cells to complete (~2-3 minutes)

**Manual execution:**
- Run cells sequentially from top to bottom

**Note:** If you encounter NumPy-related errors, go to **Runtime → Restart runtime**, then run all cells again.


In [ ]:
# Setup: Install dependencies in correct order
import sys, subprocess

# Install NumPy first (critical for binary compatibility)
print("📦 Installing NumPy...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy==1.26.4"])

# Then install packages that depend on NumPy
print("📦 Installing remaining dependencies...")
pkgs = ["pandas==2.2.2", "matplotlib==3.8.4", "scipy==1.12.0", "statsmodels==0.14.2"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

# Verify installations
print("\n✅ Verifying installations...")
import numpy, pandas, scipy, statsmodels, matplotlib
print(f"  numpy: {numpy.__version__}")
print(f"  pandas: {pandas.__version__}")
print(f"  scipy: {scipy.__version__}")
print(f"  statsmodels: {statsmodels.__version__}")
print(f"  matplotlib: {matplotlib.__version__}")
print("\n✅ All dependencies installed successfully!")


In [ ]:
# 2) Pull the repo so we have data/ and figures/ locally
import os, shutil, subprocess
REPO_URL = 'https://github.com/Course-Correct-Labs/recursive-confabulation.git'
REPO_DIR = '/content/recursive-confabulation'

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR])
print('Repo cloned to', REPO_DIR)
os.makedirs('/content/figures', exist_ok=True)

In [ ]:
# 3) Load CSVs
import pandas as pd, os
DATA_DIR = os.path.join(REPO_DIR, 'data')
expected = ['harm_irr.csv', 'intervention_effects.csv', 'significance_matrix.csv']
available = [f for f in expected if os.path.exists(os.path.join(DATA_DIR, f))]
print('Found CSVs:', available)
dfs = {name: pd.read_csv(os.path.join(DATA_DIR, name)) for name in available}
for k,v in dfs.items():
    print(k, v.shape)
assert len(dfs) >= 1, 'No expected CSVs found under data/. Make sure data files are pushed to the repo.'

In [ ]:
# 4) Statistical validation: recompute p from reported chi2 (df=1) and compare
import pandas as pd
import numpy as np
from scipy.stats import chi2 as chi2_dist

sm = dfs['significance_matrix.csv'].copy()
required = {'comparison','arm1','arm2','chi2','p_value','p_corrected'}
missing = [c for c in required if c not in sm.columns]
assert not missing, f"significance_matrix.csv missing columns: {missing}"

# Recompute p from chi2 with df=1
sm['p_from_chi2'] = 1 - chi2_dist.cdf(sm['chi2'].astype(float), df=1)

# Compare with provided p_value (tolerance for float rounding)
def close(a, b, tol=1e-8):
    return np.isfinite(a) & np.isfinite(b) & (np.abs(a - b) <= tol)

sm['p_matches'] = close(sm['p_from_chi2'].values, sm['p_value'].astype(float).values, tol=5e-4)
report_cols = ['comparison','arm1','arm2','chi2','p_value','p_from_chi2','p_corrected','significant','p_matches']
print("\n📊 χ² Validation Results:")
print(sm[report_cols].to_string(index=False))

# Quick summary
total = len(sm)
ok = sm['p_matches'].sum()
print(f"\n✅ χ²→p recomputation match: {ok}/{total} rows within 5e-4 tolerance.")
if ok == total:
    print("   All statistical tests validated successfully!")


In [ ]:
# 5) Intervention effects summary (matches manuscript claims)
import pandas as pd

ie = dfs['intervention_effects.csv'].copy()
print("\n📈 Intervention Effects Overview")
print("Columns:", list(ie.columns))
print("\nRaw data:")
print(ie.to_string(index=False))

# Check for expected structure
has_model = 'model' in ie.columns
has_arm = 'arm' in ie.columns
effect_cols = [c for c in ['effect','delta','diff','lift','effect_size'] if c in ie.columns]

if has_model and has_arm and effect_cols:
    effect_col = effect_cols[0]
    print(f"\n📊 Effects by Model and Arm (using '{effect_col}' column):")
    summary = ie.groupby(['model','arm'])[effect_col].agg(['mean','median','count'])
    print(summary.to_string())
    
    # Directionality summary
    inc = (ie[effect_col] > 0).sum()
    dec = (ie[effect_col] < 0).sum()
    print(f"\n✅ Positive effects: {inc}, Negative effects: {dec}")
else:
    print("\nNote: Standard summary not possible with current schema.")


In [ ]:
# 6) IRR sanity checks (agreement and Cohen's κ)
import pandas as pd
import numpy as np

def irr_report(df, name):
    # Look for dual-coder columns
    coder_cols = [c for c in df.columns if any(x in c.lower() for x in ['coder','rater','label_a','label_b','rater1','rater2'])]
    if len(coder_cols) < 2:
        print(f"\n{name}: Cannot find two coder columns (found: {coder_cols[:3]}...)")
        print(f"  Columns available: {list(df.columns)[:10]}")
        return
    
    a, b = coder_cols[0], coder_cols[1]
    agree = (df[a] == df[b]).mean()
    print(f"\n{name}:")
    print(f"  Comparing: {a} vs {b}")
    print(f"  Percent agreement: {agree:.3f}")
    
    # Try Cohen's kappa
    try:
        from sklearn.metrics import cohen_kappa_score
        kappa = cohen_kappa_score(df[a], df[b])
        print(f"  Cohen's κ: {kappa:.3f}")
    except Exception as e:
        print(f"  Cohen's κ: Not computed ({type(e).__name__})")

print("\n🔍 Inter-Rater Reliability Checks")
for fname in ['harm_irr.csv', 'elaboration_irr.csv', 'blame_irr.csv']:
    if fname in dfs:
        irr_report(dfs[fname], fname)
    else:
        print(f"\n{fname}: Not found in loaded data")


In [ ]:
# 7) Smoke-test figure and artifacts summary
import matplotlib.pyplot as plt
import pandas as pd
import os, glob

# Generate a simple validation plot
df_name = next(iter(dfs))
df = dfs[df_name]

fig, ax = plt.subplots(figsize=(10, 6))
if len(df.columns) > 0:
    # Plot first column value counts
    df.iloc[:, 0].astype(str).value_counts().head(10).plot(kind='bar', ax=ax)
    ax.set_title(f'Smoke Test: {df_name} - First Column Distribution')
    ax.set_xlabel('Values')
    ax.set_ylabel('Count')
else:
    ax.text(0.5, 0.5, 'No data to plot', ha='center', va='center')

out_path = '/content/figures/smoke_test_plot.png'
plt.tight_layout()
plt.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'\n✅ Saved: {out_path}')

# List all artifacts
artifacts = sorted(glob.glob('/content/figures/*'))
print(f'\n📁 Artifacts in /content/figures/ ({len(artifacts)} files):')
for art in artifacts:
    size_kb = os.path.getsize(art) / 1024
    print(f'   {os.path.basename(art)} ({size_kb:.1f} KB)')

print('\n✅ Reproduction complete!')
